# FACI: Causal Recovery Beyond the Read-out Token

**Token-site experiment.** Compare equal-size, one-vector patches at the classifier read-out token, the changed hypothesis token, and a preselected unchanged hypothesis content token. See `REPRODUCING.md` for setup and paper-layout figure instructions.

## Run in this order

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (another available GPU is acceptable for a new run).
2. Run **Cell 1** once. Then choose **Runtime → Restart session**.
3. Run **Cells 2–12 in order**. Cell 2 asks to mount your Google Drive. No uploaded source ZIP is required.
4. Cell 5 automatically freezes the same 100 eligible human-authored pairs for both models before any classifier inference. It stops if fewer than 80 distinct premise contexts qualify.
5. Cells 9 and 10 run BERT and RoBERTa separately. Completed pairs are saved immediately. Cell 11 analyzes only complete runs; Cell 12 downloads the result ZIP.
6. Keep the downloaded **FACI_Token_Sites_v1_review.zip**. It contains the new measurements, checks, and analysis. Download the executed notebook through **File → Download → Download .ipynb** to retain displayed outputs.

On interruption: reconnect with the **same GPU type**, run Cell 1 only if the packages need installation, restart after installation, and run Cells 2 onward. Completed pairs are verified and reused. A changed environment or definition stops resumption rather than mixing measurements. Do not delete a freeze file to bypass an error. Cell 12 can export a clearly marked partial diagnostic ZIP.

**No favorable result is assumed.** Do not change the cohort, threshold, locations, block window, or estimator after inspecting outputs. This is a controlled diagnostic dataset, not a general MNLI performance estimate.

## Frozen scientific question

Does patching the changed hypothesis-token state transfer more of an entailment/contradiction contrast than patching the read-out state during the first six transformer blocks?

- Two audited MNLI checkpoint revisions; FP32, math SDPA, fixed entailment-minus-contradiction logits.
- Human counterfactually augmented SNLI development and test data. Same premise; one changed lexical token; identical lengths and masks within each tokenizer. No training-data fallback, truncation, accuracy selection, or model-output-based sample selection.
- These are human-labelled contrasts; benchmark judgments can be ambiguous. Their labels are retained without claiming that every contradiction is logically exclusive or removing examples after seeing model outputs.
- The same lexical comparison word is used for both models. It is an unchanged content-like hypothesis word selected by a fixed lexical rule. It is **not** presumed causally irrelevant.
- Patch one complete residual vector after each of 12 blocks, independently at each site; entailment donor → contradiction recipient. Full-state and self patches are implementation checks only.
- Gap filter: absolute margin gap ≥0.01, exact zero excluded. Negative gaps and prediction errors remain in the primary sample if gap-valid.
- **Primary:** for each pair, average `(edited recovery − read-out recovery)` over blocks 1–6; report its median across valid pairs, separately by model. Edited-minus-comparison is secondary.
- Curves are medians across this new cohort, with 5,000 paired group-bootstrap pointwise 95% intervals. They are not the original paper's ten-seed profile estimator. No per-layer p-value search.
- Final-block read-out recovery=1 and other-site recovery=0 are architectural endpoints. Neither they nor the embedding layer enter the primary comparison.
- Coverage checks are reporting rules, not power calculations. Passing them does not by itself establish a reliable or important result. Per-model intervals do not establish a multiplicity-adjusted joint hypothesis test.
- Distinct premise contexts do not guarantee statistical independence; any dependence remaining from the original SNLI sources is outside the premise-group bootstrap.
- Equal vector counts do not ensure equal displacement; raw margin changes and patch L2 norms are saved for inspection.

## Sources and attribution

- Kaushik, Hovy, and Lipton (2020), [Learning the Difference that Makes a Difference with Counterfactually-Augmented Data](https://arxiv.org/abs/1909.12434), ICLR. [Author repository](https://github.com/acmi-lab/counterfactually-augmented-data), pinned to `6f232a1d2a11462a30ce08fb4825b734ab30828e`.
- Bowman et al. (2015), [A large annotated corpus for learning natural language inference](https://aclanthology.org/D15-1075/), EMNLP. [SNLI](https://nlp.stanford.edu/projects/snli/).
- The CAD repository declares Apache-2.0; upstream SNLI text is CC BY-SA 4.0. Source notices and attribution are retained in the export. Do not describe the underlying text as authored for this experiment.
- [Transformers 4.51.3 BERT](https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/bert) and [RoBERTa](https://huggingface.co/docs/transformers/v4.51.3/en/model_doc/roberta) documentation.


## Cell 1. Install dependencies, then restart

Run once. Colab supplies PyTorch; its exact version and GPU are recorded and guarded. Unrelated preinstalled packages may print dependency-conflict notices; subsequent import and calibration checks must pass.


In [ ]:
# Run this cell ONCE, then restart the session and start at Cell 2.
import subprocess, sys
requirements = [
    "transformers==4.51.3", "tokenizers==0.21.4", "huggingface-hub==0.34.4",
    "numpy==2.1.3", "pandas==2.2.3", "matplotlib==3.10.5", "tqdm==4.67.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--only-binary=:all:", *requirements])
_FACI_INSTALL_RESTART_REQUIRED = True
print("Installation complete. Choose Runtime > Restart session.")
print("After restart, run Cell 2 onward. Do not rerun this installation cell in the same session.")


## Cell 2. Mount Drive and freeze the runtime

Creates only `MyDrive/FACI_Token_Sites_v1`. Existing correction and precision folders are untouched.


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys, json, hashlib, random, gc, platform, warnings, time, zipfile
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

assert not globals().get("_FACI_INSTALL_RESTART_REQUIRED", False), (
    "Installation completed. Choose Runtime > Restart session, then start at Cell 2."
)
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch, transformers, tokenizers, huggingface_hub
from transformers import AutoTokenizer
from google.colab import drive, files
from tqdm.auto import tqdm

assert transformers.__version__ == "4.51.3"
assert tokenizers.__version__ == "0.21.4"
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > T4 GPU (or another GPU)."
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
torch.use_deterministic_algorithms(True)
torch.set_grad_enabled(False)

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/FACI_Token_Sites_v1")
ROOT.mkdir(parents=True, exist_ok=True)

def json_clean(x):
    if isinstance(x, dict): return {str(k): json_clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [json_clean(v) for v in x]
    if isinstance(x, np.ndarray): return json_clean(x.tolist())
    if isinstance(x, np.generic): return json_clean(x.item())
    if isinstance(x, Path): return str(x)
    if isinstance(x, float) and not np.isfinite(x):
        raise ValueError("Refusing to serialize a nonfinite number. Use None for undefined quantities.")
    return x

def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + "." + uuid4().hex + ".tmp")
    temp.write_text(json.dumps(json_clean(obj), indent=2, allow_nan=False), encoding="utf-8")
    temp.replace(path)

def sha_json(obj):
    return hashlib.sha256(json.dumps(json_clean(obj), sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()

def freeze_json(path, obj):
    path = Path(path)
    obj = json_clean(obj)
    if path.exists():
        assert json.loads(path.read_text()) == obj, (
            f"Frozen value differs at {path}. Do not delete this guard or mix runs. "
            "Preserve the old run and investigate the difference."
        )
    else:
        atomic_json(path, obj)

def remember_cell(name):
    source = get_ipython().history_manager.input_hist_raw[-1]
    freeze_json(ROOT / "notebook_cells" / (name + ".json"), {"source": source})

ENVIRONMENT = {
    "python": platform.python_version(), "torch": str(torch.__version__),
    "transformers": transformers.__version__, "tokenizers": tokenizers.__version__,
    "huggingface_hub": huggingface_hub.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "dtype": "float32", "attention": "sdpa_math", "tf32": False,
    "deterministic_algorithms": True,
}
freeze_json(ROOT / "environment.json", ENVIRONMENT)
remember_cell("02_setup")
print("New experiment folder:", ROOT)
print(json.dumps(ENVIRONMENT, indent=2))


## Cell 3. Freeze the scientific protocol

This runs before weights, predictions, and patching outcomes are loaded.


In [ ]:
SEED = 20260928
MODEL_SPECS = {
    "bert_nli": {
        "checkpoint": "textattack/bert-base-uncased-MNLI",
        "revision": "3a97f689528cbd91bcc71ab29ea6c20c089d8f28",
        "ent": 1, "con": 0, "neutral": 2,
    },
    "roberta_nli": {
        "checkpoint": "textattack/roberta-base-MNLI",
        "revision": "6f2e633322381bc5897405e417ec531ea3633a3f",
        "ent": 2, "con": 0, "neutral": 1,
    },
}
PROTOCOL = {
    "experiment": "FACI token-site counterfactual NLI", "version": 1,
    "direction": "forward_entailment_to_contradiction",
    "models": MODEL_SPECS,
    "dataset": "acmi-lab/counterfactually-augmented-data",
    "dataset_commit": "6f232a1d2a11462a30ce08fb4825b734ab30828e",
    "dataset_splits": ["dev", "test"],
    "selection": "fixed hash order; one E/C contrast per original source group; at most one pair per normalized premise; identical cohort for both models",
    "target_pairs": 100, "minimum_token_eligible_pairs": 80,
    "max_tokens": 256, "truncation": False, "selection_seed": SEED,
    "eligibility": "one whitespace-word substitution, one changed lexical token in each tokenizer, equal input lengths/masks; common unchanged hypothesis comparison word",
    "sites": ["cls", "edited", "control"],
    "patch": "replace one entire residual vector after a transformer block",
    "blocks_zero_based": list(range(12)),
    "output_score": "FP32 entailment logit minus contradiction logit; FP64 gap and recovery arithmetic",
    "gamma": 0.01, "gap_rule": "abs(gap)>=gamma and gap!=0; no accuracy or sign filtering",
    "primary_blocks_zero_based": list(range(6)),
    "primary_pair_statistic": "mean over blocks0..5 of (R_edited - R_cls)",
    "primary_population_statistic": "median over valid pairs",
    "secondary_contrast": "same first-six-block paired statistic for edited minus control",
    "curve_statistic": "median over the same valid pairs at each site/block",
    "uncertainty": "5000 paired group-bootstrap resamples; all sites and layers kept together; pointwise95% curve intervals",
    "bootstrap_replicates": 5000,
    "reporting": "descriptive curves require >=3 valid pairs and VPR>=0.5; primary interpretation also requires >=30 valid pairs and >=30 distinct premise-context groups",
    "predictions": "record all admitted pairs; neither both-correct nor gap sign determines selection",
    "calibration": "full self-patch/site/block checks on first two selected pairs; endpoints on every patched pair; full-state first/middle/final only as checks",
    "endpoints": "final cls R=1; final noncls R=0 by architecture; no embeddings or final block in primary comparison",
    "scope": "controlled diagnostic cohort, not a general MNLI accuracy estimate or replacement of previous experiments",
    "code_payload_sha256": "17fc512106879e43b2cce508565813c15ebbffd1a06107efdf6ce90a68d0fd73",
}
freeze_json(ROOT / "protocol.json", PROTOCOL)
remember_cell("03_protocol")
print("Protocol frozen before any classifier output or patching result.")


## Cell 4. Define the pinned source parser and alignment rules

The code below reads data only. Nothing from the external repository is executed.


In [ ]:
"""Pinned human-authored CAD NLI contrasts; no model-based selection.

Source: Kaushik, Hovy, and Lipton (ICLR 2020),
https://github.com/acmi-lab/counterfactually-augmented-data
Repository commit: 6f232a1d2a11462a30ce08fb4825b734ab30828e.
Repository license: Apache-2.0 (saved alongside source files).

Original SNLI examples and their two revised hypotheses occur in aligned
row groups. Every alignment and label triple is checked before pair creation.
Only the predeclared dev+test partitions are used. No training fallback.
"""
from __future__ import annotations
import base64
import csv
import hashlib
import io
import json
from pathlib import Path
import urllib.request

CAD_REPOSITORY = "acmi-lab/counterfactually-augmented-data"
CAD_REVISION = "6f232a1d2a11462a30ce08fb4825b734ab30828e"
CAD_SPLITS = ("dev", "test")
CAD_SHA256 = {
    "README.md": "b0a2750450a99e60fe392fd01a55af22a7b2f24a09281546b514728db6ac7e9e",
    "LICENSE": "c71d239df91726fc519c6eb72d318ec65820627232b2f796219e87dcf35d0ab4",
    "NLI/original/dev.tsv": "27ef042f87b7181c79609b56734caed8fb9c09ba2627d0fe876f7248141fdff7",
    "NLI/original/test.tsv": "07154b31ea7cf1bb0ac33e37bec143b928edbf2c964a73988973e5c0fe039b07",
    "NLI/revised_hypothesis/dev.tsv": "9111cba776aa125900e8c2bae73db86540398d735cfcf7f9f15d49bb5ec20d31",
    "NLI/revised_hypothesis/test.tsv": "eff28a3c8c4a99150a81b36ad1c4840a9ea91ad0f488e514b1c143d12a4032b5",
}
CAD_EXPECTED_ROWS = {
    "NLI/original/dev.tsv": 200, "NLI/original/test.tsv": 400,
    "NLI/revised_hypothesis/dev.tsv": 400,
    "NLI/revised_hypothesis/test.tsv": 800,
}
CAD_CITATION = {
    "authors": ["Divyansh Kaushik", "Eduard Hovy", "Zachary C. Lipton"],
    "title": "Learning the Difference that Makes a Difference with Counterfactually Augmented Data",
    "venue": "International Conference on Learning Representations (ICLR)",
    "year": 2020,
    "paper": "https://arxiv.org/abs/1909.12434",
    "repository": "https://github.com/" + CAD_REPOSITORY,
    "revision": CAD_REVISION,
    "repository_license": "Apache-2.0",
    "upstream_task": "SNLI",
    "upstream_license": "Creative Commons Attribution-ShareAlike 4.0 International",
    "upstream_homepage": "https://nlp.stanford.edu/projects/snli/",
    "license_note": "Repository LICENSE and upstream SNLI attribution are both retained; repository license is not a relicensing claim about upstream text.",
}


def _download_source(relative_path: str) -> bytes:
    raw = f"https://raw.githubusercontent.com/{CAD_REPOSITORY}/{CAD_REVISION}/{relative_path}"
    api = f"https://api.github.com/repos/{CAD_REPOSITORY}/contents/{relative_path}?ref={CAD_REVISION}"
    errors = []
    for url in (raw, api):
        for attempt in range(2):
            try:
                request = urllib.request.Request(url, headers={"User-Agent": "FACI-token-site-experiment/1"})
                with urllib.request.urlopen(request, timeout=45) as response:
                    data = response.read()
                if url == api:
                    obj = json.loads(data)
                    if obj.get("encoding") != "base64":
                        raise ValueError("Unexpected GitHub content encoding")
                    data = base64.b64decode(obj["content"])
                actual = hashlib.sha256(data).hexdigest()
                expected = CAD_SHA256[relative_path]
                if actual != expected:
                    raise ValueError(f"Source digest mismatch: {relative_path}: {actual} != {expected}")
                return data
            except Exception as error:
                errors.append(f"{url}, attempt {attempt + 1}: {type(error).__name__}: {error}")
    raise RuntimeError("Cannot fetch pinned source; no alternate dataset was substituted.\n" + "\n".join(errors))


def acquire_cad_sources(cache_dir: str | Path) -> tuple[dict[str, Path], dict]:
    """Verify cached bytes or download the exact author-provided files."""
    root = Path(cache_dir)
    root.mkdir(parents=True, exist_ok=True)
    paths = {}
    files = []
    for relative, expected in CAD_SHA256.items():
        path = root / relative
        if path.exists():
            data = path.read_bytes()
            if hashlib.sha256(data).hexdigest() != expected:
                raise ValueError(f"Cached source changed: {path}. Move this cache aside and rerun.")
        else:
            data = _download_source(relative)
            path.parent.mkdir(parents=True, exist_ok=True)
            partial = path.with_suffix(path.suffix + ".partial")
            partial.write_bytes(data)
            partial.replace(path)
        paths[relative] = path
        files.append({"path": relative, "sha256": expected, "bytes": len(data),
                      "url": f"https://raw.githubusercontent.com/{CAD_REPOSITORY}/{CAD_REVISION}/{relative}"})
    manifest = {"citation": CAD_CITATION, "source_files": files,
                "partitions": list(CAD_SPLITS), "training_data_used": False,
                "pairing": "Original row i with revised-hypothesis rows 2*i and 2*i+1 (zero-based); validated premise and E/C/N labels."}
    return paths, manifest


def _read_tsv(path: Path, relative: str) -> list[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle, delimiter="\t")
        if reader.fieldnames != ["sentence1", "sentence2", "gold_label"]:
            raise ValueError(f"Unexpected columns in {relative}: {reader.fieldnames}")
        rows = list(reader)
    if len(rows) != CAD_EXPECTED_ROWS[relative]:
        raise ValueError(f"Unexpected row count in {relative}: {len(rows)}")
    if any(None in row or any(value is None for value in row.values()) for row in rows):
        raise ValueError(f"Malformed TSV record in {relative}")
    return rows


def load_cad_contrasts(cache_dir: str | Path) -> tuple[list[dict], dict]:
    """Return all 600 fixed E/C contrasts; token eligibility is a later step.

    `original_row` and the rows in `source_rows` are zero-based data row indices
    excluding the header. Source text is preserved, including whitespace.
    `group_id` marks each original/revision family. `premise_cluster_id` groups
    identical (whitespace-normalized) premises if a later analysis needs to
    account for repeated premise contexts. One E/C contrast exists per family;
    an originally neutral item contributes its two human revisions.
    """
    paths, manifest = acquire_cad_sources(cache_dir)
    pairs = []
    split_counts = {}
    for split in CAD_SPLITS:
        original_name = f"NLI/original/{split}.tsv"
        revised_name = f"NLI/revised_hypothesis/{split}.tsv"
        originals = _read_tsv(paths[original_name], original_name)
        revised = _read_tsv(paths[revised_name], revised_name)
        if len(revised) != 2 * len(originals):
            raise ValueError(f"Unexpected revision alignment in {split}")
        for index, original in enumerate(originals):
            group = [(original, original_name, index),
                     (revised[2 * index], revised_name, 2 * index),
                     (revised[2 * index + 1], revised_name, 2 * index + 1)]
            if len({row["sentence1"] for row, _, _ in group}) != 1:
                raise ValueError(f"Premise alignment failed at {split}/{index}")
            labels = [row["gold_label"] for row, _, _ in group]
            if sorted(labels) != ["contradiction", "entailment", "neutral"]:
                raise ValueError(f"Label alignment failed at {split}/{index}: {labels}")
            by_label = {row["gold_label"]: (row, file, row_index) for row, file, row_index in group}
            ent, con = by_label["entailment"], by_label["contradiction"]
            premise = original["sentence1"]
            family = f"cad_{split}_{index:04d}"
            pairs.append({
                "id": family, "pair_id": family, "group_id": family, "source_split": split,
                "original_row": index, "original_label": original["gold_label"],
                "premise": premise,
                "premise_cluster_id": "premise_" + hashlib.sha256(" ".join(premise.split()).encode()).hexdigest()[:16],
                "entailment_hypothesis": ent[0]["sentence2"],
                "contradiction_hypothesis": con[0]["sentence2"],
                "source_rows": {
                    "entailment": {"file": ent[1], "row_zero_based": ent[2]},
                    "contradiction": {"file": con[1], "row_zero_based": con[2]},
                },
            })
        split_counts[split] = len(originals)
    if len({row["id"] for row in pairs}) != len(pairs):
        raise ValueError("Duplicate contrast identifiers")
    manifest["contrast_count"] = len(pairs)
    manifest["contrasts_per_split"] = split_counts
    manifest["unique_premise_clusters"] = len({row["premise_cluster_id"] for row in pairs})
    manifest["selection_uses_model_outputs"] = False
    return pairs, manifest



"""Prospective, output-blind alignment and site selection for FACI token sites."""
import hashlib
import re


# A fixed lexical exclusion list identifies content-like comparison words.
# This is a lexical rule, not a claim that the selected vector has zero effect.
CONTROL_STOPWORDS = frozenset(
    "a an the this that these those i you he she it we they me him her us them "
    "my your his its our their mine yours hers ours theirs is am are was were "
    "be been being have has had do does did and or but if then than so as at by "
    "for from in into of on onto to with without not no nor all any some each "
    "every either neither both one two three there here who whom whose which "
    "what when where why how can could may might must shall should will would".split()
)


def stable_rank(seed, key):
    return hashlib.sha256(f"{seed}|{key}".encode()).hexdigest()


def single_word_edit(pair):
    a = list(re.finditer(r"\S+", pair["entailment_hypothesis"]))
    b = list(re.finditer(r"\S+", pair["contradiction_hypothesis"]))
    if len(a) != len(b):
        return None, "word_count_differs"
    diff = [i for i, (x, y) in enumerate(zip(a, b)) if x.group() != y.group()]
    if len(diff) != 1:
        return None, "not_exactly_one_whitespace_word_substitution"
    i = diff[0]
    if not any(c.isalpha() for c in a[i].group()) or not any(c.isalpha() for c in b[i].group()):
        return None, "changed_word_is_not_alphabetic"
    return {"word_index": i, "ent_span": a[i].span(), "con_span": b[i].span(),
            "ent_word": a[i].group(), "con_word": b[i].group()}, None


def tokenize_aligned(pair, tokenizer, max_length=256):
    edit, error = single_word_edit(pair)
    if error:
        return None, error
    encodings = []
    for key in ("entailment_hypothesis", "contradiction_hypothesis"):
        enc = tokenizer(pair["premise"], text_pair=pair[key],
                        truncation=False, padding=False,
                        return_offsets_mapping=True, return_special_tokens_mask=True)
        if len(enc["input_ids"]) > max_length:
            return None, "too_long_no_truncation_allowed"
        if tokenizer.unk_token_id in enc["input_ids"]:
            return None, "unknown_token"
        encodings.append(enc)
    a, b = encodings
    if len(a["input_ids"]) != len(b["input_ids"]):
        return None, "token_count_differs"
    for field in ("attention_mask", "token_type_ids", "special_tokens_mask"):
        if a.get(field) != b.get(field):
            return None, f"{field}_differs"
    sa, sb = a.sequence_ids(), b.sequence_ids()
    if sa != sb:
        return None, "sequence_assignment_differs"
    differing = [i for i, (u, v) in enumerate(zip(a["input_ids"], b["input_ids"])) if u != v]
    if len(differing) != 1:
        return None, "not_exactly_one_token_difference"
    changed = differing[0]
    if changed == 0 or sa[changed] != 1 or a["special_tokens_mask"][changed]:
        return None, "changed_token_is_not_hypothesis_content"
    if a["input_ids"][0] != tokenizer.cls_token_id:
        return None, "unexpected_readout_token"
    # Require a whole changed whitespace word represented by one token.
    for enc, span in zip(encodings, (edit["ent_span"], edit["con_span"])):
        overlaps = [i for i, ((lo, hi), seq) in enumerate(zip(enc["offset_mapping"], enc.sequence_ids()))
                    if seq == 1 and hi > span[0] and lo < span[1] and hi > lo]
        # Punctuation may form another unchanged token in the same word;
        # handle lexical alphabetic span instead of dropping every period.
        raw = pair["entailment_hypothesis"] if enc is a else pair["contradiction_hypothesis"]
        core = re.search(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", raw[span[0]:span[1]])
        if core is None:
            return None, "no_changed_lexical_core"
        core_span = (span[0] + core.start(), span[0] + core.end())
        core_tokens = [i for i, ((lo, hi), seq) in enumerate(zip(enc["offset_mapping"], enc.sequence_ids()))
                       if seq == 1 and hi > core_span[0] and lo < core_span[1] and hi > lo]
        if core_tokens != [changed]:
            return None, "changed_word_has_multiple_subtokens"
    controls = []
    words_a = list(re.finditer(r"\S+", pair["entailment_hypothesis"]))
    words_b = list(re.finditer(r"\S+", pair["contradiction_hypothesis"]))
    for word_index, (wa, wb) in enumerate(zip(words_a, words_b)):
        if word_index == edit["word_index"] or wa.group() != wb.group():
            continue
        core = re.search(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", wa.group())
        if core is None or core.group().lower() in CONTROL_STOPWORDS:
            continue
        core_span = (wa.start() + core.start(), wa.start() + core.end())
        token_ids = [i for i, ((lo, hi), seq) in enumerate(zip(a["offset_mapping"], sa))
                     if seq == 1 and hi > core_span[0] and lo < core_span[1] and hi > lo]
        if len(token_ids) != 1:
            continue
        idx = token_ids[0]
        if idx in (0, changed) or a["special_tokens_mask"][idx] or a["input_ids"][idx] != b["input_ids"][idx]:
            continue
        # Check the same word is one token at the same index in recipient.
        b_span = (wb.start() + core.start(), wb.start() + core.end())
        bt = [i for i, ((lo, hi), seq) in enumerate(zip(b["offset_mapping"], sb))
              if seq == 1 and hi > b_span[0] and lo < b_span[1] and hi > lo]
        if bt == [idx]:
            controls.append({"word_index": word_index, "token_index": idx, "word": wa.group()})
    inputs = []
    for enc in encodings:
        inputs.append({k: [list(enc[k])] for k in tokenizer.model_input_names if k in enc})
    return {"donor_inputs": inputs[0], "recipient_inputs": inputs[1],
            "changed_token": changed, "edit": edit, "control_candidates": controls,
            "donor_tokens": tokenizer.convert_ids_to_tokens(a["input_ids"]),
            "recipient_tokens": tokenizer.convert_ids_to_tokens(b["input_ids"])}, None


def select_common_cohort(candidates, tokenizers, seed=20260928, target=100, minimum=80, max_length=256):
    eligible, audit = [], []
    seen_groups, seen_text = set(), set()
    for pair in sorted(candidates, key=lambda p: stable_rank(seed, p["pair_id"])):
        row = {"pair_id": pair["pair_id"], "group_id": pair["group_id"], "eligible": False,
               "reason": None, "model_reasons": {}}
        aligned = {}
        for key, tok in tokenizers.items():
            aligned[key], reason = tokenize_aligned(pair, tok, max_length)
            row["model_reasons"][key] = reason
        if any(v is None for v in aligned.values()):
            row["reason"] = "token_or_word_alignment"
        else:
            shared = set.intersection(*[
                {x["word_index"] for x in a["control_candidates"]} for a in aligned.values()
            ])
            if not shared:
                row["reason"] = "no_common_single_token_comparison_word"
            else:
                control_word = min(shared, key=lambda i: stable_rank(seed, f"{pair['pair_id']}|control|{i}"))
                for a in aligned.values():
                    chosen = next(x for x in a["control_candidates"] if x["word_index"] == control_word)
                    a["positions"] = {"cls": 0, "edited": a["changed_token"], "control": chosen["token_index"]}
                    a["comparison_word"] = chosen["word"]
                text_key = (pair["premise"].strip(), pair["entailment_hypothesis"].strip(),
                            pair["contradiction_hypothesis"].strip())
                if pair["group_id"] in seen_groups or text_key in seen_text:
                    row["reason"] = "duplicate_group_or_text"
                else:
                    seen_groups.add(pair["group_id"])
                    seen_text.add(text_key)
                    row["eligible"] = True
                    eligible.append({**pair, "alignment": aligned})
        audit.append(row)
    selected = eligible[:target]
    chosen_ids = {p["pair_id"] for p in selected}
    for a in audit:
        a["selected"] = a["pair_id"] in chosen_ids
    report = {"n_candidates": len(candidates), "n_eligible": len(eligible), "n_selected": len(selected),
              "target": target, "minimum": minimum, "selection_seed": seed,
              "ready": len(selected) >= minimum, "audit": audit}
    return selected, report

remember_cell("04_dataset_logic")
print("Pinned source parser and output-blind eligibility rules ready.")


## Cell 5. Select and freeze the common cohort

Downloads verified source bytes and tokenizers. Saves every eligibility decision plus `data/selected_pairs.csv`. The original row families and exact source rows remain traceable.


In [ ]:
# Source selection happens before model weights or classifier outputs are loaded.
source_pairs, SOURCE_MANIFEST = load_cad_contrasts(ROOT / "data" / "source")
freeze_json(ROOT / "data" / "source_manifest.json", SOURCE_MANIFEST)
for p in source_pairs:
    p["source_family_id"] = p["group_id"]
    # Keep at most one contrast for an identical normalized premise context.
    p["group_id"] = p["premise_cluster_id"]

TOKENIZERS = {}
for key, spec in MODEL_SPECS.items():
    tok = AutoTokenizer.from_pretrained(spec["checkpoint"], revision=spec["revision"],
                                       use_fast=True, trust_remote_code=False)
    assert tok.is_fast
    TOKENIZERS[key] = tok

COHORT, COHORT_AUDIT = select_common_cohort(
    source_pairs, TOKENIZERS, seed=SEED,
    target=PROTOCOL["target_pairs"], minimum=PROTOCOL["minimum_token_eligible_pairs"],
    max_length=PROTOCOL["max_tokens"],
)
freeze_json(ROOT / "data" / "cohort_audit.json", COHORT_AUDIT)
freeze_json(ROOT / "data" / "cohort.json", COHORT)
COHORT = json.loads((ROOT / "data" / "cohort.json").read_text())
assert len({p["group_id"] for p in COHORT}) == len(COHORT)
assert COHORT_AUDIT["ready"], (
    f"Only {len(COHORT)} eligible distinct premise-context pairs. Need at least "
    f"{PROTOCOL['minimum_token_eligible_pairs']} under the frozen protocol. "
    "Stop before inference. Do not relax the rule or substitute training data."
)
cohort_table = pd.DataFrame([
    {"pair_id": p["pair_id"], "group_id": p["group_id"], "source_split": p["source_split"],
     "premise": p["premise"], "entailment_hypothesis": p["entailment_hypothesis"],
     "contradiction_hypothesis": p["contradiction_hypothesis"],
     "changed_ent_word": p["alignment"]["bert_nli"]["edit"]["ent_word"],
     "changed_con_word": p["alignment"]["bert_nli"]["edit"]["con_word"],
     "comparison_word": p["alignment"]["bert_nli"]["comparison_word"]}
    for p in COHORT
])
cohort_table.to_csv(ROOT / "data" / "selected_pairs.csv", index=False)
remember_cell("05_prepare_cohort")
print(f"Source contrasts: {COHORT_AUDIT['n_candidates']}; eligible unique contexts: "
      f"{COHORT_AUDIT['n_eligible']}; selected for BOTH models: {len(COHORT)}.")
print("No classifier outputs or patching outcomes were used in selection.")
display(cohort_table.head(8))


## Cell 6. Define the patch engine and runtime checks

Hooks run at transformer-block outputs. All hooks are removed even if inference fails. Missing head weights or incompatible shapes stop execution.


In [ ]:
"""Strict, single-pair activation-patching engine for the FACI token-site study.

No old FACI modules are imported or modified. All result objects are JSON-safe.
Block numbers in results are 1-based; index 0 denotes the first transformer block,
never the embedding output. Interventions replace outputs of complete blocks.
"""
from __future__ import annotations

import math
from typing import Mapping

import torch


VERIFIED_SPECS = {
    "bert_nli": {
        "checkpoint": "textattack/bert-base-uncased-MNLI",
        "revision": "3a97f689528cbd91bcc71ab29ea6c20c089d8f28",
        "ent": 1, "con": 0, "neutral": 2,
    },
    "roberta_nli": {
        "checkpoint": "textattack/roberta-base-MNLI",
        "revision": "6f2e633322381bc5897405e417ec531ea3633a3f",
        "ent": 2, "con": 0, "neutral": 1,
    },
}

SITES = ("cls", "edited", "control")
ATOL = 1e-5
RTOL = 1e-5


def _require(ok, message):
    if not ok:
        raise RuntimeError(message)


def _blocks(model, expected_layers=12):
    kind = model.config.model_type
    _require(kind in ("bert", "roberta"), f"Unsupported model type: {kind}")
    blocks = list(getattr(model, kind).encoder.layer)
    _require(len(blocks) == expected_layers,
             f"Expected {expected_layers} complete blocks, got {len(blocks)}")
    return blocks


def _tensor_error(actual, expected, name, atol=ATOL, rtol=RTOL):
    _require(tuple(actual.shape) == tuple(expected.shape), f"{name}: shape mismatch")
    error = float((actual.float() - expected.float()).abs().max().item())
    _require(torch.allclose(actual, expected, atol=atol, rtol=rtol),
             f"{name} failed: maximum absolute error {error:.9g}")
    return error


def load_verified_nli(model_key: str, device: str = "cuda"):
    """Return (model, tokenizer, provenance); reject absent/newly initialized weights.

    Uses only the two explicitly audited checkpoint revisions and manual label
    mappings. Generic LABEL_n config names are recorded, never interpreted.
    """
    import transformers
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    _require(transformers.__version__ == "4.51.3",
             f"Expected transformers 4.51.3, found {transformers.__version__}")
    _require(model_key in VERIFIED_SPECS, f"Unverified model: {model_key}")
    spec = dict(VERIFIED_SPECS[model_key])
    tokenizer = AutoTokenizer.from_pretrained(
        spec["checkpoint"], revision=spec["revision"], use_fast=True,
        trust_remote_code=False,
    )
    _require(tokenizer.is_fast, "Fast tokenizer required for sequence/offset alignment")
    model, info = AutoModelForSequenceClassification.from_pretrained(
        spec["checkpoint"], revision=spec["revision"],
        output_loading_info=True, torch_dtype=torch.float32,
        attn_implementation="sdpa", trust_remote_code=False,
    )
    allowed_unexpected = {
        "roberta.pooler.dense.weight", "roberta.pooler.dense.bias"
    } if model_key == "roberta_nli" else set()
    _require(not info.get("missing_keys"), f"Missing checkpoint weights: {info}")
    _require(not info.get("mismatched_keys"), f"Mismatched checkpoint weights: {info}")
    _require(not info.get("error_msgs"), f"Checkpoint loading errors: {info}")
    _require(set(info.get("unexpected_keys", [])) <= allowed_unexpected,
             f"Unexpected checkpoint weights: {info}")
    _require(model.config.num_labels == 3, "Expected a three-class NLI head")
    expected_class = "BertForSequenceClassification" if model_key == "bert_nli" else "RobertaForSequenceClassification"
    _require(type(model).__name__ == expected_class,
             f"Expected {expected_class}, got {type(model).__name__}")
    _blocks(model)
    # If config labels have explicit semantics, they must agree with the audited map.
    names = {"entailment": "ent", "contradiction": "con", "neutral": "neutral"}
    for index, label in model.config.id2label.items():
        label = str(label).strip().lower()
        if label in names:
            _require(int(index) == spec[names[label]],
                     f"Config label contradicts audited map: {index}={label}")
    # Match the bounded correction's SDPA math attention recipe, never flash.
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    if hasattr(torch.backends.cuda, "enable_cudnn_sdp"):
        torch.backends.cuda.enable_cudnn_sdp(False)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    model.to(device=device, dtype=torch.float32).eval()
    provenance = {
        "model_key": model_key, **spec,
        "model_class": type(model).__name__,
        "tokenizer_class": type(tokenizer).__name__,
        "config_id2label": {str(k): str(v) for k, v in model.config.id2label.items()},
        "loading_info": info,
        "label_mapping_source": "audited bounded NLI correction, fixed checkpoint revision",
        "dtype": "float32", "attention": "sdpa_math", "tf32": False,
        "transformers": transformers.__version__,
    }
    return model, tokenizer, provenance


def smoke_check_labels(model, tokenizer, label_indices: Mapping[str, int]):
    """Check three fixed semantic examples; never fit or permute the label mapping.

    A failure is a diagnostic stop, not grounds to search for passing examples.
    The examples are not used in the experiment or its statistical summaries.
    """
    premise = "A person is eating an apple."
    examples = [
        ("ent", "A person is eating."),
        ("con", "No one is eating."),
        ("neutral", "The apple was grown in Italy."),
    ]
    rows = []
    device = next(model.parameters()).device
    model.eval()
    with torch.inference_mode():
        for label, hypothesis in examples:
            inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=False)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            logits = model(**inputs, return_dict=True).logits.detach().float().cpu()
            pred = int(logits.argmax(dim=-1).item())
            rows.append({"premise": premise, "hypothesis": hypothesis,
                         "expected_label": label, "expected_index": int(label_indices[label]),
                         "predicted_index": pred, "logits": logits[0].tolist(),
                         "passed": pred == int(label_indices[label])})
    _require(all(row["passed"] for row in rows),
             f"Fixed label smoke check failed. Do not remap labels automatically: {rows}")
    return rows


def _prepare_pair(model, donor_inputs, recipient_inputs, positions):
    _require(not model.training, "Model must be in eval mode")
    _require(not torch.is_autocast_enabled(), "Disable autocast for this FP32 experiment")
    _require(next(model.parameters()).dtype == torch.float32, "FP32 model required")
    allowed = {"input_ids", "attention_mask", "token_type_ids"}
    _require(set(donor_inputs) == set(recipient_inputs), "Paired input keys differ")
    _require(set(donor_inputs) <= allowed, "Unexpected input keys")
    _require({"input_ids", "attention_mask"} <= set(donor_inputs), "Missing required inputs")
    device = next(model.parameters()).device
    donor = {k: v.detach().to(device) for k, v in donor_inputs.items()}
    recipient = {k: v.detach().to(device) for k, v in recipient_inputs.items()}
    shape = tuple(donor["input_ids"].shape)
    _require(len(shape) == 2 and shape[0] == 1, "Exactly one pair per call is required")
    for key in donor:
        _require(tuple(donor[key].shape) == shape == tuple(recipient[key].shape),
                 f"Full paired shape mismatch for {key}")
        if key != "input_ids":
            _require(torch.equal(donor[key], recipient[key]), f"Paired {key} differ")
    _require(bool((donor["attention_mask"] == 1).all().item()),
             "Pass unpadded, untruncated paired sequences; masks must be all ones")
    _require(set(positions) == set(SITES), f"Sites must be exactly {SITES}")
    pos = {k: int(v) for k, v in positions.items()}
    _require(pos["cls"] == 0, "Classifier read-out must be position zero")
    _require(len(set(pos.values())) == 3, "Site positions must be distinct")
    _require(all(0 <= value < shape[1] for value in pos.values()), "Site out of range")
    changed = (donor["input_ids"] != recipient["input_ids"]).nonzero(as_tuple=False)
    _require(tuple(changed.shape) == (1, 2), "Expected exactly one changed token ID")
    _require(int(changed[0, 1].item()) == pos["edited"], "Edited site does not match changed ID")
    _require(int(donor["input_ids"][0, 0].item()) == int(model.config.bos_token_id or 0)
             if model.config.model_type == "roberta" else True,
             "RoBERTa read-out token is not its BOS token")
    return donor, recipient, pos


def _forward_with_cache(model, inputs, blocks, verify_hidden_states=False):
    cache, counts, handles = {}, [0] * len(blocks), []

    def make_hook(index):
        def hook(module, args, output):
            _require(isinstance(output, tuple) and len(output) >= 1,
                     "Transformer block must return a tuple")
            hidden = output[0]
            _require(hidden.ndim == 3 and hidden.shape[0] == 1,
                     "Expected block output shape [1, sequence, hidden]")
            counts[index] += 1
            cache[index] = hidden.detach().float().cpu().clone()
        return hook

    try:
        for index, block in enumerate(blocks):
            handles.append(block.register_forward_hook(make_hook(index)))
        output = model(**inputs, output_hidden_states=verify_hidden_states, return_dict=True)
    finally:
        for handle in handles:
            handle.remove()
    _require(counts == [1] * len(blocks), f"Cache hooks must each fire once: {counts}")
    logits = output.logits.detach().float().cpu()
    _require(tuple(logits.shape) == (1, 3), f"Unexpected logit shape {tuple(logits.shape)}")
    _require(bool(torch.isfinite(logits).all().item()), "Nonfinite baseline logits")
    cache_errors = []
    if verify_hidden_states:
        _require(len(output.hidden_states) == len(blocks) + 1,
                 "Hidden states must contain embeddings plus one entry per block")
        for index in range(len(blocks)):
            cache_errors.append(_tensor_error(
                cache[index], output.hidden_states[index + 1].detach().float().cpu(),
                f"Block {index + 1} hook vs hidden_states[{index + 1}]"))
    return logits, cache, cache_errors


def _patched_forward(model, inputs, block, replacement, position):
    """Return CPU logits, L2 intervention size, and relative L2 intervention size.

    position=None is a whole-state calibration. The tensor replacement must have
    the full [1, sequence, hidden] shape even for a single-position intervention.
    """
    observed = {"calls": 0}

    def patch_hook(module, args, output):
        _require(isinstance(output, tuple) and len(output) >= 1,
                 "Transformer block must return a tuple")
        hidden = output[0]
        _require(tuple(hidden.shape) == tuple(replacement.shape), "Full patch shape mismatch")
        _require(hidden.ndim == 3 and hidden.shape[0] == 1, "Invalid patch batch shape")
        _require(bool(torch.isfinite(hidden).all().item()), "Nonfinite recipient hidden state")
        source = replacement.to(device=hidden.device, dtype=hidden.dtype)
        _require(bool(torch.isfinite(source).all().item()), "Nonfinite donor hidden state")
        result = hidden.clone()
        if position is None:
            before, after = hidden, source
            result = source.clone()
        else:
            before, after = hidden[:, position, :], source[:, position, :]
            result[:, position, :] = after
        observed["calls"] += 1
        observed["l2"] = float(torch.linalg.vector_norm(after - before).item())
        recipient_norm = float(torch.linalg.vector_norm(before).item())
        observed["recipient_norm"] = recipient_norm
        observed["relative_l2"] = observed["l2"] / recipient_norm if recipient_norm > 0 else None
        return (result,) + output[1:]

    handle = block.register_forward_hook(patch_hook)
    try:
        logits = model(**inputs, return_dict=True).logits.detach().float().cpu()
    finally:
        handle.remove()
    _require(observed["calls"] == 1, f"Patch hook fired {observed['calls']} times")
    _require(tuple(logits.shape) == (1, 3), "Unexpected patched logit shape")
    _require(bool(torch.isfinite(logits).all().item()), "Nonfinite patched logits")
    return logits, observed


def _margin(logits, labels):
    # Preserve the corrected FACI arithmetic: FP32 subtraction, then Python float.
    return float((logits[0, int(labels["ent"])] - logits[0, int(labels["con"])]).item())


@torch.inference_mode()
def run_site_pair(model, donor_inputs, recipient_inputs, positions,
                  label_indices, gap_threshold=0.01, calibrate=False,
                  expected_layers=12):
    """Patch the donor into recipient at all blocks and cls/edited/control sites.

    Inputs are tokenized NLI dictionaries with unpadded batch size one, identical
    masks/segment IDs and exactly one changed input ID. ``positions`` uses keys
    cls/edited/control. Site selection and hypothesis-token validation belong to
    the frozen dataset preparation, not this engine.

    Return keys: donor_logits, recipient_logits, donor_margin, recipient_margin,
    gap, retained, exclusion_reason, donor_predicted_index,
    recipient_predicted_index, positions, patches, calibration, endpoint_checks.

    ``patches`` has 36 rows for a retained 12-block pair and zero for a rejected
    gap; no model correctness filter is applied. Each row includes block (1..12),
    site, patched_logits, patched_margin, raw_delta, recovery, intervention_l2,
    recipient_vector_l2, relative_intervention_l2. Recovery is signed and unclipped.

    calibrate=True additionally verifies all cached hidden states, self-patches
    every block/site, and replaces full states at first/middle/final blocks.
    Calibrations run even for a gap-rejected pair. Every retained pair checks final
    CLS=donor and final non-CLS=recipient logits; these are architectural endpoints.
    """
    _require(math.isfinite(float(gap_threshold)) and gap_threshold >= 0, "Invalid gap threshold")
    _require(set(label_indices) >= {"ent", "con"}, "Explicit entailment/contradiction indices required")
    _require(int(label_indices["ent"]) != int(label_indices["con"]), "Contrast labels must differ")
    _require(all(0 <= int(label_indices[k]) < 3 for k in ("ent", "con")), "Label index out of range")
    blocks = _blocks(model, expected_layers)
    donor, recipient, pos = _prepare_pair(model, donor_inputs, recipient_inputs, positions)
    dlogits, dcache, dc_errors = _forward_with_cache(model, donor, blocks, calibrate)
    rlogits, rcache, rc_errors = _forward_with_cache(model, recipient, blocks, calibrate)
    dm, rm = _margin(dlogits, label_indices), _margin(rlogits, label_indices)
    gap = dm - rm
    retained = math.isfinite(gap) and gap != 0 and abs(gap) >= gap_threshold
    result = {
        "donor_logits": dlogits[0].tolist(), "recipient_logits": rlogits[0].tolist(),
        "donor_margin": dm, "recipient_margin": rm, "gap": gap,
        "retained": retained, "exclusion_reason": None if retained else "gap_below_threshold_or_zero",
        "donor_predicted_index": int(dlogits.argmax(dim=-1).item()),
        "recipient_predicted_index": int(rlogits.argmax(dim=-1).item()),
        "positions": pos, "gap_threshold": float(gap_threshold),
        "patches": [], "calibration": None, "endpoint_checks": {},
    }
    if calibrate:
        checks = {"donor_cache_max_abs_errors": dc_errors,
                  "recipient_cache_max_abs_errors": rc_errors,
                  "self_patch": [], "whole_state": []}
        for index, block in enumerate(blocks):
            for site in SITES:
                logits, norms = _patched_forward(model, recipient, block, rcache[index], pos[site])
                error = _tensor_error(logits, rlogits, f"Self patch block={index + 1} site={site}")
                _require(norms["l2"] == 0.0, "Self patch must inject a zero L2 change")
                checks["self_patch"].append({"block": index + 1, "site": site,
                                              "max_abs_logit_error": error,
                                              "logits": logits[0].tolist(),
                                              "intervention_l2": norms["l2"]})
        for index in sorted({0, (len(blocks) - 1) // 2, len(blocks) - 1}):
            logits, norms = _patched_forward(model, recipient, blocks[index], dcache[index], None)
            error = _tensor_error(logits, dlogits, f"Whole-state patch block={index + 1}")
            checks["whole_state"].append({"block": index + 1, "max_abs_logit_error": error,
                                            "logits": logits[0].tolist(),
                                            "intervention_l2": norms["l2"]})
        result["calibration"] = checks
    if not retained:
        return result
    for index, block in enumerate(blocks):
        for site in SITES:
            logits, norms = _patched_forward(model, recipient, block, dcache[index], pos[site])
            margin = _margin(logits, label_indices)
            raw_delta = margin - rm
            result["patches"].append({
                "block": index + 1, "site": site,
                "patched_logits": logits[0].tolist(), "patched_margin": margin,
                "raw_delta": raw_delta, "recovery": raw_delta / gap,
                "intervention_l2": norms["l2"],
                "recipient_vector_l2": norms["recipient_norm"],
                "relative_intervention_l2": norms["relative_l2"],
            })
            if index == len(blocks) - 1:
                target = dlogits if site == "cls" else rlogits
                result["endpoint_checks"][site] = _tensor_error(
                    logits, target, f"Final-block architecture endpoint {site}")
    return result

assert VERIFIED_SPECS == MODEL_SPECS
remember_cell("06_engine")
print("One-vector patch engine ready.")


## Cell 7. Freeze the analysis implementation

Defines the estimator, reporting rules, bootstrap intervals, and figures before measurement.


In [ ]:
"""Frozen analysis for the FACI single-token site comparison.

Input interface
---------------
records: DataFrame or list of dicts, one row per valid pair x block x site.
    Required: model, pair_id, group_id, direction, block, site,
              recovery, numerator, gap, valid, both_correct, patch_l2.
    block is zero-based; site is one of cls/edited/control; direction=forward.
pair_status: DataFrame or list of dicts, one row per admitted pair x model.
    Required: model, pair_id, group_id, direction, gap, valid, both_correct.
    Contains invalid-gap pairs too. All admitted pairs must appear for both models.

Recovery is signed numerator / signed gap. No clipping or accuracy filtering.
This module never loads a model, changes a prompt, or selects a favorable result.
"""

from pathlib import Path
import json
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ANALYSIS_SITES = ("cls", "edited", "control")
ANALYSIS_SITE_LABELS = {
    "cls": "Read-out token",
    "edited": "Changed hypothesis token",
    "control": "Unchanged content token",
}
ANALYSIS_SITE_COLORS = {"cls": "#31688e", "edited": "#cc6235", "control": "#54875b"}
ANALYSIS_KEYS = ["model", "pair_id", "direction"]
ANALYSIS_PRIMARY_BLOCKS = tuple(range(6))
ANALYSIS_PROTOCOL = {
    "design": "New counterfactual NLI diagnostic set; one-vector patch at each site",
    "direction": "forward: entailment donor into contradiction recipient",
    "n_blocks": 12,
    "block_index_in_records": "zero-based transformer-block outputs; no embeddings",
    "gamma": 0.01,
    "validity": "finite gap != 0 and abs(gap) >= gamma; no accuracy filtering",
    "primary_blocks_zero_based": list(ANALYSIS_PRIMARY_BLOCKS),
    "primary_pair_score": "mean over blocks 0..5 of R_edited - R_cls",
    "primary_estimate": "median of primary_pair_score over valid pairs, per model",
    "secondary_pair_score": "mean over blocks 0..5 of R_edited - R_control",
    "raw_numerator_pair_score": "mean over blocks 0..5 of numerator_edited - numerator_cls",
    "curve_estimate": "median over the same complete set of valid pairs at each block and site",
    "uncertainty": "percentile 95% bootstrap intervals; resample source groups with replacement, retaining all pairs and all sites/blocks in each selected group",
    "curve_interval_scope": "pointwise, not simultaneous; no layerwise hypothesis tests",
    "minimum_pairs_before_inference": 80,
    "target_pairs_before_inference": 100,
    "minimum_valid_pairs_for_descriptive_applicability": 3,
    "minimum_valid_pair_ratio": 0.5,
    "minimum_valid_pairs_for_primary_interpretation": 30,
    "minimum_valid_groups_for_primary_interpretation": 30,
    "correctness_strata": "descriptive only; do not change the primary sample",
    "endpoints": "final-block CLS patch is forced donor recovery; final-block other-token patches cannot influence a CLS-only classifier",
    "limitations": "minimal counterfactual set, single-token interventions, and two checkpoints; not a replication of the original ten-seed layer-profile estimator; per-model intervals do not adjust for multiple models or secondary comparisons",
}


def _analysis_bool_column(series, name):
    if series.isna().any():
        raise ValueError(f"{name} contains missing boolean values")
    if not series.map(lambda x: isinstance(x, (bool, np.bool_)) or x in (0, 1)).all():
        raise ValueError(f"{name} must contain booleans or 0/1, not strings")
    return series.astype(bool)


def _analysis_require(df, names, title):
    missing = sorted(set(names) - set(df.columns))
    if missing:
        raise ValueError(f"{title} is missing columns: {missing}")
    if df[names].isna().any().any():
        cols = df[names].columns[df[names].isna().any()].tolist()
        raise ValueError(f"{title} contains missing required values: {cols}")


def _analysis_clean_inputs(records, pair_status, gamma):
    rec = pd.DataFrame(records).copy()
    status = pd.DataFrame(pair_status).copy()
    status_required = ANALYSIS_KEYS + ["group_id", "gap", "valid", "both_correct"]
    _analysis_require(status, status_required, "pair_status")
    rec_required = status_required + ["block", "site", "recovery", "numerator", "patch_l2"]
    if not rec.empty:
        _analysis_require(rec, rec_required, "records")
    else:
        rec = pd.DataFrame(columns=rec_required)
    for df in (rec, status):
        for col in ["model", "pair_id", "group_id", "direction"]:
            df[col] = df[col].astype(str)
        for col in ["valid", "both_correct"]:
            df[col] = _analysis_bool_column(df[col], col)
        df["gap"] = pd.to_numeric(df["gap"], errors="raise")
        if not np.isfinite(df["gap"].to_numpy(float)).all():
            raise ValueError("Nonfinite model output/gap must stop the run; do not silently omit the pair")
    if status.empty or status[ANALYSIS_KEYS].duplicated().any():
        raise ValueError("pair_status must contain one row per model/pair/direction")
    if set(status["direction"]) != {"forward"}:
        raise ValueError("This frozen analysis is forward-only")
    models = sorted(status["model"].unique())
    if len(models) != 2:
        raise ValueError("The frozen comparison requires exactly the two prespecified models")
    base_ids = None
    base_groups = None
    for model in models:
        sm = status[status["model"] == model]
        ids = set(sm["pair_id"])
        groups = sm.set_index("pair_id")["group_id"].to_dict()
        if base_ids is None:
            base_ids, base_groups = ids, groups
        elif ids != base_ids or groups != base_groups:
            raise ValueError("Both models must be evaluated on exactly the same frozen admitted pairs/groups")
    derived = status["gap"].ne(0) & status["gap"].abs().ge(gamma)
    if not derived.equals(status["valid"]):
        raise ValueError("valid differs from the prespecified absolute-gap rule")
    if not rec.empty:
        if set(rec["site"]) - set(ANALYSIS_SITES):
            raise ValueError("Unknown patch site")
        if rec[ANALYSIS_KEYS + ["block", "site"]].duplicated().any():
            raise ValueError("Duplicate patch outcomes")
        rec["block"] = pd.to_numeric(rec["block"], errors="raise")
        if not rec["block"].isin(range(12)).all():
            raise ValueError("Expected zero-based block indices 0..11")
        rec["block"] = rec["block"].astype(int)
        for col in ["recovery", "numerator", "patch_l2"]:
            rec[col] = pd.to_numeric(rec[col], errors="raise")
            if not np.isfinite(rec[col].to_numpy(float)).all():
                raise ValueError(f"Nonfinite {col}; stop, diagnose, and retain failed attempt")
        if rec["patch_l2"].lt(0).any():
            raise ValueError("Patch L2 cannot be negative")
        joined = rec.merge(status[status_required], on=ANALYSIS_KEYS, how="left", suffixes=("", "_status"), validate="many_to_one", indicator=True)
        if not joined["_merge"].eq("both").all():
            raise ValueError("Patch record lacks a corresponding pair_status row")
        for col in ["group_id", "valid", "both_correct"]:
            if not joined[col].eq(joined[col + "_status"]).all():
                raise ValueError(f"Patch/status mismatch: {col}")
        if not np.allclose(joined["gap"], joined["gap_status"], rtol=1e-10, atol=1e-12):
            raise ValueError("Patch/status gap mismatch")
        if not rec["valid"].all():
            raise ValueError("Patch records must contain only gap-valid pairs; retain invalid pairs in pair_status")
        if not np.allclose(rec["recovery"], rec["numerator"] / rec["gap"], rtol=1e-7, atol=1e-9):
            raise ValueError("Recovery is not signed numerator / signed gap")
    counts = rec.groupby(ANALYSIS_KEYS, observed=True).size().to_dict()
    for row in status.itertuples(index=False):
        k = (row.model, row.pair_id, row.direction)
        expected = 36 if row.valid else 0
        if counts.get(k, 0) != expected:
            raise ValueError(f"Incomplete/misclassified patch outcomes for {k}: expected {expected}, got {counts.get(k, 0)}. Do not analyze a partial run.")
    return rec, status, models


def _analysis_bootstrap_indices(group_ids, n_boot, rng):
    groups = np.asarray(group_ids, dtype=str)
    unique = np.unique(groups)
    positions = [np.flatnonzero(groups == g) for g in unique]
    for _ in range(n_boot):
        sampled = rng.integers(0, len(unique), size=len(unique))
        yield np.concatenate([positions[i] for i in sampled])


def _analysis_percentile_interval(values):
    return np.quantile(np.asarray(values, dtype=float), [0.025, 0.975]).tolist()


def _analysis_json_safe(value):
    if isinstance(value, dict):
        return {str(k): _analysis_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_analysis_json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def analyze_site_records(records, pair_status, outdir, seed=20260928, n_boot=5000):
    """Validate completeness, save all outcomes, estimate frozen contrasts, plot.

    Returns dict with summary, contrasts, curves, and paths. Does not interpret
    a positive/negative interval automatically or select extra analyses.
    """
    if int(n_boot) < 1000:
        raise ValueError("Use at least 1,000 bootstrap replicates; frozen default is 5,000")
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    rec, status, models = _analysis_clean_inputs(records, pair_status, ANALYSIS_PROTOCOL["gamma"])
    rec.to_csv(outdir / "patch_outcomes.csv", index=False)
    status.to_csv(outdir / "pair_status.csv", index=False)
    metadata = dict(ANALYSIS_PROTOCOL)
    metadata.update({"bootstrap_seed": int(seed), "bootstrap_replicates": int(n_boot), "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "quantile_method": "linear", "model_order": models})
    (outdir / "analysis_protocol.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    rng = np.random.default_rng(seed)
    summaries, contrast_rows, curve_rows, stratum_rows = [], [], [], []
    boot_arrays = {}
    for model in models:
        sm = status[status["model"] == model].sort_values("pair_id")
        sv = sm[sm["valid"]]
        n, nv, ng = len(sm), len(sv), sv["group_id"].nunique()
        vpr = nv / n
        descriptive = nv >= 3 and vpr >= 0.5
        primary_allowed = n >= 80 and descriptive and nv >= 30 and ng >= 30
        summary = {
            "model": model, "n_admitted": n, "n_groups_admitted": int(sm["group_id"].nunique()),
            "n_valid": nv, "n_valid_groups": int(ng), "valid_pair_ratio": vpr,
            "n_positive_gap": int(sm["gap"].gt(0).sum()), "n_negative_gap": int(sm["gap"].lt(0).sum()),
            "n_exact_zero_gap": int(sm["gap"].eq(0).sum()),
            "n_nonzero_below_gamma": int((sm["gap"].ne(0) & sm["gap"].abs().lt(0.01)).sum()),
            "both_correct_fraction_all_admitted": float(sm["both_correct"].mean()),
            "both_correct_fraction_valid": float(sv["both_correct"].mean()) if nv else None,
            "descriptive_applicability_gate": bool(descriptive),
            "primary_interpretation_gate": bool(primary_allowed),
            "gate_note": "Passes prespecified size/applicability checks; evaluate estimate and limitations" if primary_allowed else "INSUFFICIENT PRESPECIFIED EVIDENCE: show counts/diagnostics, do not make a primary empirical claim or lower thresholds",
        }
        # Optional independently recorded per-side correctness is reported over
        # the entire admitted set, never used to choose the primary sample.
        for correctness in ["donor_correct", "recipient_correct"]:
            if correctness in sm.columns:
                vals = _analysis_bool_column(sm[correctness], correctness)
                summary[correctness + "_fraction_all_admitted"] = float(vals.mean())
        if nv == 0:
            summary.update({"primary_median_delta": None, "primary_ci95": None, "secondary_median_delta": None, "secondary_ci95": None, "raw_primary_median_delta": None})
            summaries.append(summary)
            continue
        rm = rec[rec["model"] == model]
        ids = sv["pair_id"].tolist()
        ordered_cols = pd.MultiIndex.from_product([range(12), ANALYSIS_SITES], names=["block", "site"])
        tensors = {}
        for value in ["recovery", "numerator", "patch_l2"]:
            wide = rm.pivot(index="pair_id", columns=["block", "site"], values=value).reindex(index=ids, columns=ordered_cols)
            if wide.isna().any().any():
                raise ValueError("Missing site/block outcomes after pivot; no partial-case analysis")
            tensors[value] = wide.to_numpy(float).reshape(nv, 12, 3)
        R, N, L2 = tensors["recovery"], tensors["numerator"], tensors["patch_l2"]
        # site order is cls, edited, control.
        primary = np.mean(R[:, ANALYSIS_PRIMARY_BLOCKS, 1] - R[:, ANALYSIS_PRIMARY_BLOCKS, 0], axis=1)
        secondary = np.mean(R[:, ANALYSIS_PRIMARY_BLOCKS, 1] - R[:, ANALYSIS_PRIMARY_BLOCKS, 2], axis=1)
        raw = np.mean(N[:, ANALYSIS_PRIMARY_BLOCKS, 1] - N[:, ANALYSIS_PRIMARY_BLOCKS, 0], axis=1)
        for i, row in enumerate(sv.itertuples(index=False)):
            contrast_rows.append({"model": model, "pair_id": row.pair_id, "group_id": row.group_id, "direction": "forward", "gap": float(row.gap), "both_correct": bool(row.both_correct), "primary_mean_delta_edited_minus_cls_blocks1to6": float(primary[i]), "secondary_mean_delta_edited_minus_control_blocks1to6": float(secondary[i]), "raw_mean_delta_edited_minus_cls_blocks1to6": float(raw[i])})
        boot_primary, boot_secondary = np.empty(n_boot), np.empty(n_boot)
        boot_curves = np.empty((n_boot, 12, 3))
        for b, indices in enumerate(_analysis_bootstrap_indices(sv["group_id"].tolist(), n_boot, rng)):
            boot_primary[b] = np.median(primary[indices])
            boot_secondary[b] = np.median(secondary[indices])
            boot_curves[b] = np.median(R[indices], axis=0)
        # One source group provides no between-source bootstrap information.
        # Preserve the computed diagnostic but do not present a degenerate CI.
        meaningful_ci = ng >= 2
        lo, hi = np.quantile(boot_curves, [0.025, 0.975], axis=0)
        median_R, median_N, median_L2 = (np.median(tensors[k], axis=0) for k in ["recovery", "numerator", "patch_l2"])
        for block in range(12):
            for si, site in enumerate(ANALYSIS_SITES):
                curve_rows.append({"model": model, "direction": "forward", "block": block, "block_number": block + 1, "site": site, "n_valid": nv, "n_valid_groups": int(ng), "median_recovery": float(median_R[block, si]), "pointwise_ci95_low": float(lo[block, si]) if meaningful_ci else None, "pointwise_ci95_high": float(hi[block, si]) if meaningful_ci else None, "median_raw_numerator": float(median_N[block, si]), "median_patch_l2": float(median_L2[block, si]), "architectural_endpoint": block == 11})
        endpoint = {
            "cls_max_absolute_recovery_error_from_one": float(np.max(np.abs(R[:, -1, 0] - 1))),
            "other_sites_max_absolute_recovery_from_zero": float(np.max(np.abs(R[:, -1, 1:]))),
            "cls_max_absolute_numerator_minus_gap": float(np.max(np.abs(N[:, -1, 0] - sv["gap"].to_numpy(float)))),
            "other_sites_max_absolute_numerator": float(np.max(np.abs(N[:, -1, 1:]))),
        }
        endpoint_pass = np.allclose(N[:, -1, 0], sv["gap"].to_numpy(float), rtol=1e-4, atol=1e-5) and np.all(np.abs(N[:, -1, 1:]) <= 1e-5)
        endpoint["check_pass"] = bool(endpoint_pass)
        if not endpoint_pass:
            primary_allowed = False
            summary["primary_interpretation_gate"] = False
            summary["gate_note"] = "ARCHITECTURAL ENDPOINT CHECK FAILED: diagnose the intervention/read-out implementation before interpreting any effect"
        summary.update({"primary_median_delta": float(np.median(primary)), "primary_ci95": _analysis_percentile_interval(boot_primary) if meaningful_ci else None, "secondary_median_delta": float(np.median(secondary)), "secondary_ci95": _analysis_percentile_interval(boot_secondary) if meaningful_ci else None, "raw_primary_median_delta": float(np.median(raw)), "endpoint_audit": endpoint})
        for correct, label in [(True, "both_correct"), (False, "not_both_correct")]:
            mask = sv["both_correct"].to_numpy(bool) == correct
            stratum_rows.append({"model": model, "stratum": label, "n_valid": int(mask.sum()), "n_valid_groups": int(sv.loc[mask, "group_id"].nunique()), "primary_median_delta_descriptive_only": float(np.median(primary[mask])) if mask.any() else None, "raw_primary_median_delta_descriptive_only": float(np.median(raw[mask])) if mask.any() else None, "note": "No resampling inference or new selection; primary analysis includes all gap-valid pairs"})
        boot_arrays[model.replace("/", "__") + "__primary"] = boot_primary
        boot_arrays[model.replace("/", "__") + "__secondary"] = boot_secondary
        summaries.append(summary)
    summary_df, contrasts, curves, strata = map(pd.DataFrame, [summaries, contrast_rows, curve_rows, stratum_rows])
    summary_df.to_csv(outdir / "model_summary.csv", index=False)
    contrasts.to_csv(outdir / "pair_contrasts.csv", index=False)
    curves.to_csv(outdir / "layer_site_summary.csv", index=False)
    strata.to_csv(outdir / "correctness_strata_descriptive.csv", index=False)
    summary_payload = {"protocol": metadata, "models": summaries}
    (outdir / "analysis_summary.json").write_text(json.dumps(_analysis_json_safe(summary_payload), indent=2, allow_nan=False), encoding="utf-8")
    np.savez_compressed(outdir / "bootstrap_contrast_estimates.npz", **boot_arrays)
    if not curves.empty:
        _analysis_plot_curves(curves, summaries, models, outdir)
    notes = _analysis_report(summaries)
    (outdir / "RESULTS_README.md").write_text(notes, encoding="utf-8")
    return {"summary": summary_df, "contrasts": contrasts, "curves": curves, "strata": strata, "paths": sorted(str(p) for p in outdir.iterdir() if p.is_file())}


def _analysis_plot_curves(curves, summaries, models, outdir):
    lookup = {m["model"]: m for m in summaries}
    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42})
    # Recovery panels use identical y-limits so visual magnitude is comparable;
    # values and intervals are never truncated to [0,1].
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.35), sharey=True)
    for ax, model in zip(axes, models):
        sub = curves[curves["model"] == model]
        sm = lookup[model]
        ax.axhline(0, color="#999999", linewidth=0.8)
        ax.axhline(1, color="#bbbbbb", linewidth=0.7, linestyle=":")
        ax.axvline(6.5, color="#aaaaaa", linewidth=0.8, linestyle="--")
        ax.axvspan(11.65, 12.35, color="#dddddd", alpha=0.5)
        for site in ANALYSIS_SITES:
            s = sub[sub["site"] == site].sort_values("block")
            ax.plot(s["block_number"], s["median_recovery"], color=ANALYSIS_SITE_COLORS[site], marker="o", markersize=3, label=ANALYSIS_SITE_LABELS[site])
            ax.fill_between(s["block_number"].to_numpy(float), s["pointwise_ci95_low"].to_numpy(float), s["pointwise_ci95_high"].to_numpy(float), color=ANALYSIS_SITE_COLORS[site], alpha=0.14)
        display_model = model.split("/")[-1]
        ax.set_title(f"{display_model}\n{sm['n_valid']}/{sm['n_admitted']} valid pairs; {sm['n_valid_groups']} source groups", fontsize=10)
        ax.set_xticks([1, 3, 6, 9, 12])
        ax.set_xlim(0.65, 12.35)
        ax.set_xlabel("Transformer block (1-based)")
        if not sm["primary_interpretation_gate"]:
            ax.text(0.02, 0.98, "Interpretation gate failed", transform=ax.transAxes, va="top", fontsize=8, color="#a43f34")
    axes[0].set_ylabel("Median signed recovery")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 0.03), ncol=3, frameon=False, fontsize=9)
    fig.text(0.5, 0.012, "Bands: pointwise 95% source-group bootstrap intervals. Blocks 1–6: prespecified contrast. Shading at block 12: architectural endpoints.", ha="center", fontsize=7.5)
    fig.tight_layout(rect=(0, 0.14, 1, 1))
    fig.savefig(outdir / "recovery_by_token_site.png", dpi=220, bbox_inches="tight")
    fig.savefig(outdir / "recovery_by_token_site.pdf", bbox_inches="tight")
    plt.close(fig)
    # Raw effect and perturbation size are saved as diagnostics so normalized
    # recovery or unequal activation displacement cannot hide the measurement.
    fig, axes = plt.subplots(2, 2, figsize=(11, 6.4), sharex=True)
    for col, model in enumerate(models):
        sub = curves[curves["model"] == model]
        for row, (metric, label) in enumerate([("median_raw_numerator", "Median raw margin change"), ("median_patch_l2", "Median patch L2 displacement")]):
            ax = axes[row, col]
            for site in ANALYSIS_SITES:
                s = sub[sub["site"] == site].sort_values("block")
                ax.plot(s["block_number"], s[metric], color=ANALYSIS_SITE_COLORS[site], marker="o", markersize=2.7, label=ANALYSIS_SITE_LABELS[site])
            ax.axhline(0, color="#bbbbbb", linewidth=0.7)
            ax.axvspan(11.65, 12.35, color="#dddddd", alpha=0.5)
            ax.set_ylabel(label)
            ax.set_xticks([1, 3, 6, 9, 12])
            if row == 0:
                ax.set_title(model.split("/")[-1], fontsize=10)
            else:
                ax.set_xlabel("Transformer block (1-based)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(outdir / "raw_margin_and_patch_norm_diagnostics.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def _analysis_report(summaries):
    lines = ["# FACI token-site experiment: frozen analysis", "", "This is a new diagnostic experiment. It does not replace, rerun, or repair the historical FACI benchmark.", "", "The primary estimate is the median across valid pairs of each pair's mean (edited-token recovery minus read-out-token recovery) across the first six transformer blocks. A positive estimate means the edited-token patch transfers more of the measured contrast within this prespecified early window. It does not locate a unique representation, prove necessity, or establish when an entire model first computes an answer.", "", "All intervals are per-model percentile 95% source-group bootstrap intervals. Curve bands are pointwise, not simultaneous. There are no per-layer p-values. Curves are median-over-pairs for this new dataset, not the historical ten-seed profile aggregation. Recovery is signed and unbounded; no clipping or correctness filtering is applied.", ""]
    for sm in summaries:
        lines += [f"## {sm['model']}", "", f"Admitted pairs: {sm['n_admitted']}; gap-valid pairs: {sm['n_valid']}; valid source groups: {sm['n_valid_groups']}; VPR: {sm['valid_pair_ratio']:.3f}.", f"Primary interpretation gate: {sm['primary_interpretation_gate']}. {sm['gate_note']}"]
        if sm["primary_median_delta"] is not None:
            lines += [f"Primary median edited-minus-read-out contrast: {sm['primary_median_delta']:.6g}; 95% CI: {sm['primary_ci95']}.", f"Secondary median edited-minus-control contrast: {sm['secondary_median_delta']:.6g}; 95% CI: {sm['secondary_ci95']}.", f"Raw margin primary contrast (descriptive median): {sm['raw_primary_median_delta']:.6g}.", f"Architectural endpoint check passed: {sm['endpoint_audit']['check_pass']}."]
        lines += [""]
    lines += ["## Interpretation limits", "", "Final-block read-out recovery is forced by the intervention architecture; final-block edited/control recovery is zero because the classifier does not read those positions after the final block. These are implementation checks, not discoveries. The first six blocks are the prespecified primary window.", "", "Larger effects at a token may partly reflect larger activation displacement. Inspect the patch-norm diagnostic; this experiment holds the number of patched vectors fixed, not their L2 displacement. The unchanged token is a comparison location, not a guaranteed null. Negative gaps and errors remain in the primary sample if gap-valid; correctness strata are descriptive only.", "", "Do not select an earlier/later block window, change gamma, drop difficult examples, switch the curve estimator, or regenerate examples based on these outcomes. An uncertain or null result remains a valid result; a failed gate means the evidence is insufficient for the planned empirical claim.", ""]
    return "\n".join(lines)

assert ANALYSIS_PROTOCOL["gamma"] == PROTOCOL["gamma"]
assert ANALYSIS_PROTOCOL["primary_blocks_zero_based"] == PROTOCOL["primary_blocks_zero_based"]
freeze_json(ROOT / "analysis_definition.json", ANALYSIS_PROTOCOL)
remember_cell("07_analysis_definition")
print("Analysis definition frozen before measurement.")


## Cell 8. Define the resumable runner

Each pair is saved only after its arithmetic and endpoint checks pass. Failed attempts are recorded and raised; they are not replaced with zeros or dropped.


In [ ]:
def frozen_run_signature():
    paths = [ROOT / "protocol.json", ROOT / "environment.json",
             ROOT / "data" / "source_manifest.json", ROOT / "data" / "cohort.json",
             ROOT / "analysis_definition.json"]
    paths += sorted((ROOT / "notebook_cells").glob("0[2345678]_*.json"))
    hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
    return sha_json(hashes), hashes


def validate_pair_result(obj, expected_context):
    assert obj["context"] == expected_context, "Stored pair belongs to a different frozen run."
    key = expected_context["model"]
    spec = MODEL_SPECS[key]
    assert obj["positions"] == expected_context["positions"]
    assert obj["gap_threshold"] == PROTOCOL["gamma"]
    d = np.asarray(obj["donor_logits"], dtype=np.float32)
    r = np.asarray(obj["recipient_logits"], dtype=np.float32)
    assert d.shape == r.shape == (3,) and np.isfinite(d).all() and np.isfinite(r).all()
    dm, rm = float(d[spec["ent"]] - d[spec["con"]]), float(r[spec["ent"]] - r[spec["con"]])
    gap = dm - rm
    assert gap == obj["gap"] and dm == obj["donor_margin"] and rm == obj["recipient_margin"]
    valid = gap != 0 and abs(gap) >= PROTOCOL["gamma"]
    assert bool(obj["retained"]) == valid
    assert obj["donor_predicted_index"] == int(d.argmax())
    assert obj["recipient_predicted_index"] == int(r.argmax())
    patches = obj["patches"]
    assert len(patches) == (36 if valid else 0)
    assert {(p["block"], p["site"]) for p in patches} == (
        {(b, s) for b in range(1, 13) for s in ("cls", "edited", "control")} if valid else set()
    )
    for patch in patches:
        z = np.asarray(patch["patched_logits"], dtype=np.float32)
        assert z.shape == (3,) and np.isfinite(z).all()
        g = float(z[spec["ent"]] - z[spec["con"]])
        assert g == patch["patched_margin"] and g - rm == patch["raw_delta"]
        assert patch["recovery"] == (g - rm) / gap
        assert np.isfinite(patch["intervention_l2"]) and patch["intervention_l2"] >= 0
        if patch["block"] == 12:
            np.testing.assert_allclose(z, d if patch["site"] == "cls" else r, rtol=1e-5, atol=1e-5)
    if expected_context["calibrate"]:
        assert len(obj["calibration"]["self_patch"]) == 36
        assert len(obj["calibration"]["whole_state"]) == 3
    return obj


def pair_context(model_key, pair, index, signature):
    return {"run_signature": signature, "model": model_key, "pair_id": pair["pair_id"],
            "group_id": pair["group_id"], "cohort_index": index,
            "pair_sha256": sha_json(pair), "direction": "forward", "calibrate": index < 2,
            "positions": pair["alignment"][model_key]["positions"]}


def run_model(model_key):
    assert model_key in MODEL_SPECS
    assert VERIFIED_SPECS == MODEL_SPECS
    signature, hashes = frozen_run_signature()
    freeze_json(ROOT / "run_signature.json", {"sha256": signature, "files": hashes})
    contexts = [pair_context(model_key, p, i, signature) for i, p in enumerate(COHORT)]
    folder = ROOT / "raw" / model_key
    folder.mkdir(parents=True, exist_ok=True)
    expected_files = [folder / (p["pair_id"] + ".json") for p in COHORT]
    if all(p.exists() for p in expected_files):
        for path, context in zip(expected_files, contexts):
            validate_pair_result(json.loads(path.read_text()), context)
        print(model_key, ": all", len(COHORT), "pair records already complete and verified; no inference repeated.")
        return
    model = None
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model, tok, provenance = load_verified_nli(model_key, device="cuda")
        freeze_json(ROOT / "models" / (model_key + ".json"), provenance)
        atomic_json(ROOT / "models" / "load_sessions" / (model_key + "_" + uuid4().hex[:8] + ".json"),
                    {"time_utc": datetime.now(timezone.utc).isoformat(),
                     "warnings": [{"category": type(w.message).__name__, "message": str(w.message)} for w in caught]})
        labels = {k: MODEL_SPECS[model_key][k] for k in ("ent", "con", "neutral")}
        try:
            label_check = smoke_check_labels(model, tok, labels)
        except Exception as error:
            atomic_json(ROOT / "checks" / (model_key + "_label_error.json"),
                        {"error": str(error), "time_utc": datetime.now(timezone.utc).isoformat()})
            raise
        freeze_json(ROOT / "checks" / (model_key + "_label_examples.json"), label_check)
        for index, (pair, path, context) in enumerate(tqdm(
            list(zip(COHORT, expected_files, contexts)), desc=model_key
        )):
            if path.exists():
                validate_pair_result(json.loads(path.read_text()), context)
                continue
            alignment = pair["alignment"][model_key]
            # Recheck live tokenizer against the frozen literal input IDs.
            again, reason = tokenize_aligned(pair, tok, PROTOCOL["max_tokens"])
            assert reason is None
            assert again["donor_inputs"] == alignment["donor_inputs"]
            assert again["recipient_inputs"] == alignment["recipient_inputs"]
            donor = {k: torch.tensor(v, dtype=torch.long) for k, v in alignment["donor_inputs"].items()}
            recipient = {k: torch.tensor(v, dtype=torch.long) for k, v in alignment["recipient_inputs"].items()}
            start = time.monotonic()
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    result = run_site_pair(model, donor, recipient, alignment["positions"], labels,
                                           gap_threshold=PROTOCOL["gamma"], calibrate=index < 2)
                result["context"] = context
                result["elapsed_seconds"] = time.monotonic() - start
                result["warnings"] = [{"category": type(w.message).__name__, "message": str(w.message)} for w in caught]
                validate_pair_result(result, context)
                atomic_json(path, result)
            except Exception as error:
                import traceback
                atomic_json(ROOT / "errors" / model_key / (pair["pair_id"] + "_" + uuid4().hex[:8] + ".json"),
                            {"context": context, "error_type": type(error).__name__, "error": str(error),
                             "traceback": traceback.format_exc(), "time_utc": datetime.now(timezone.utc).isoformat()})
                raise
        completion = {"run_signature": signature, "model": model_key, "n_pairs": len(COHORT),
                      "files": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in expected_files}}
        atomic_json(ROOT / "checks" / (model_key + "_complete.json"), completion)
        print(model_key, "completed:", len(COHORT), "saved pair records.")
    finally:
        if model is not None:
            model.to("cpu")
            del model
        gc.collect()
        torch.cuda.empty_cache()


def gather_records():
    signature, _ = frozen_run_signature()
    records, statuses = [], []
    for key, spec in MODEL_SPECS.items():
        for index, pair in enumerate(COHORT):
            path = ROOT / "raw" / key / (pair["pair_id"] + ".json")
            assert path.is_file(), f"Incomplete experiment: missing {path.name} for {key}. Resume its model cell."
            result = validate_pair_result(json.loads(path.read_text()), pair_context(key, pair, index, signature))
            dc = result["donor_predicted_index"] == spec["ent"]
            rc = result["recipient_predicted_index"] == spec["con"]
            base = {"model": key, "pair_id": pair["pair_id"], "group_id": pair["group_id"],
                    "direction": "forward", "gap": result["gap"], "valid": result["retained"],
                    "donor_correct": dc, "recipient_correct": rc, "both_correct": dc and rc}
            statuses.append(base)
            for patch in result["patches"]:
                records.append({**base, "block": patch["block"] - 1, "site": patch["site"],
                                "recovery": patch["recovery"], "numerator": patch["raw_delta"],
                                "patch_l2": patch["intervention_l2"]})
    return records, statuses


remember_cell("08_runner")
print("Runner ready. Completed pairs are verified and reused on resume.")


## Cell 9. Run BERT

The first two frozen pairs also receive full self-patch/cache/full-state checks. Every retained pair checks final-layer endpoints. Watch the progress bar; duration depends on the assigned GPU.


In [ ]:
run_model("bert_nli")


## Cell 10. Run RoBERTa

Uses the same frozen pairs, the same word-level comparison locations, and the audited RoBERTa label map.


In [ ]:
run_model("roberta_nli")


## Cell 11. Analyze and display the figure

Requires all admitted pairs for both models. Outputs include raw CSVs, prespecified contrasts, coverage, bootstrap estimates, an editable-data-backed PNG/PDF figure, and neutral interpretation notes.


In [ ]:
# This refuses to analyze a partial run or silently omit failed pairs.
records, statuses = gather_records()
analysis_result = analyze_site_records(
    records, statuses, ROOT / "analysis", seed=SEED,
    n_boot=PROTOCOL["bootstrap_replicates"],
)
display(analysis_result["summary"][[
    "model", "n_admitted", "n_valid", "valid_pair_ratio",
    "both_correct_fraction_all_admitted", "primary_median_delta",
    "primary_ci95", "primary_interpretation_gate", "gate_note",
]])
from IPython.display import Image, display, Markdown
figure = ROOT / "analysis" / "recovery_by_token_site.png"
if figure.exists(): display(Image(filename=str(figure)))
display(Markdown((ROOT / "analysis" / "RESULTS_README.md").read_text()))
remember_cell("11_summarize")


## Cell 12. Export the review bundle

Download the ZIP to preserve the complete run and its analysis. This cell can also export a PARTIAL bundle after a failure; it never labels an incomplete measurement as complete.


In [ ]:
# This cell also works after an earlier error once Cell 2 has run.
# A partial archive is explicitly marked and is useful for diagnosis.
root_files = sorted(p for p in ROOT.rglob("*") if p.is_file() and p.suffix not in {".tmp", ".partial"})
source_manifest = {str(p.relative_to(ROOT)): {
    "sha256": hashlib.sha256(p.read_bytes()).hexdigest(), "bytes": p.stat().st_size
} for p in root_files if p.name != "export_manifest.json"}
complete = False
validation_error = None
try:
    all_records, all_statuses = gather_records()
    complete = len(all_statuses) == 2 * len(COHORT)
except Exception as error:
    validation_error = str(error)
atomic_json(ROOT / "export_manifest.json", {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "measurement_complete": complete,
    "analysis_present": (ROOT / "analysis" / "analysis_summary.json").is_file(),
    "validation_error": validation_error, "files": source_manifest,
})
archive_name = "FACI_Token_Sites_v1_review.zip" if complete else "FACI_Token_Sites_v1_PARTIAL.zip"
archive = Path("/content") / archive_name
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(ROOT.rglob("*")):
        if p.is_file() and p.suffix not in {".tmp", ".partial"}:
            z.write(p, arcname=str(p.relative_to(ROOT)))
print("Measurement complete:", complete)
print("Export:", archive.name, "|", round(archive.stat().st_size / 1024**2, 2), "MB")
files.download(str(archive))
